### Commodity Group Dimension table creation

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
dbutils.widgets.text('catalog','agmarknet')


In [0]:
catalog = dbutils.widgets.get('catalog')
print(catalog)

In [0]:
print(silver_schema)

**Reading commodities groups json file from s3 and  write the df to **bronze****

In [0]:
data_path_json = f's3://agmarknet-pc/*.json'
print(data_path_json)

In [0]:
schema = StructType(
    [
        StructField("status",StringType()),
        StructField("message", StringType()),
        StructField("data", ArrayType(
            StructType([
                StructField("id",IntegerType()),
                StructField("cmdt_name",StringType()),
                StructField("group_id",IntegerType()),
                StructField("group_name",StringType()),
                StructField("status",StringType())
            ])
        ))
    ]
)

In [0]:
df_json = (
    spark.read
    .schema(schema)
    .option("multiline",True)
    .json(data_path_json)
)

In [0]:
display(df_json)

In [0]:
df_json.printSchema()

In [0]:
commodity_json_df = df_json.select(F.explode(F.col("data")).alias("commodity"))
commodity_json_df.show(truncate=False)

In [0]:
final_comm_df = commodity_json_df.select(
    "commodity.group_id",
    "commodity.group_name",
    "commodity.status"
).distinct()

final_comm_df.show(truncate=False)


In [0]:
final_comm_df.count()

In [0]:
final_comm_df.select("cmdt_name").distinct().count()

In [0]:
final_comm_df.select("group_id").distinct().count()
#df1.show()

In [0]:
final_comm_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{bronze_schema}.dim_commodity_group')

### Silver layer transformations on groups table

In [0]:
df_groups_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.dim_commodity_group")

In [0]:
df_groups_silver = df_groups_bronze \
.withColumnRenamed("group_id", "Commodity_Group_Id") \
.withColumnRenamed("group_name", "Commodity_Group_Name") \
.withColumnRenamed("status", "Status") 

In [0]:
df_groups_silver.show()

In [0]:
df_groups_silver.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("enableChangeDataFeed","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{silver_schema}.dim_commodity_group')


In [0]:
df_groups_silver.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("enableChangeDataFeed","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.dim_commodity_group')

###Creating commodity dimension table

In [0]:
df_commodity = spark.sql(f"select distinct Commodity_Code,Commodity,Variety,Grade from {catalog}.{silver_schema}.daily_prices")
df_commodity.show(30)

In [0]:
df_commodity.count()

In [0]:
df_grade = spark.sql(f"select distinct Grade from {catalog}.{silver_schema}.daily_prices")
df_grade.count()
df_grade.show()
#Commodity_Code,Commodity,Variety

In [0]:
df_commodity.write \
.format("delta") \
.option("overwriteSchema","true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.dim_commodity')

In [0]:
df_com = spark.sql(f"select * from {catalog}.{gold_schema}.dim_commodity")


In [0]:
df_com.count()

In [0]:
display(df_com)